In [1]:
import os
import json
from pathlib import Path
import pandas as pd



In [2]:
INPUT_ROOT = Path("../input/AI4Code")
if not INPUT_ROOT.exists():
    INPUT_ROOT = Path("/kaggle/input/AI4Code")
TEST_DIR = INPUT_ROOT / "test"
SAMPLE_SUBMISSION = INPUT_ROOT / "sample_submission.csv"



In [3]:
assert TEST_DIR.is_dir(), f"Test directory not found: {TEST_DIR}"
assert SAMPLE_SUBMISSION.is_file(), f"Sample submission not found: {SAMPLE_SUBMISSION}"



In [4]:
# Build predictions: keep code cells in original order, sort markdown cells by descending length,
# then interleave them (code, markdown, code, …) to produce a more plausible ordering.
predicted_orders = {}
for json_path in TEST_DIR.glob("*.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    source_dict = notebook.get("source", {})
    cell_type_dict = notebook.get("cell_type", {})

    # Fallback: if cell_type info is missing, use original key order.
    if not cell_type_dict:
        final_order = list(source_dict.keys())
    else:
        # Separate code and markdown cells.
        code_keys = [
            k
            for k, v in cell_type_dict.items()
            if isinstance(v, str) and v.lower() == "code"
        ]
        markdown_keys = [
            k
            for k, v in cell_type_dict.items()
            if isinstance(v, str) and v.lower() == "markdown"
        ]

        # Preserve original order for code cells.
        # If some keys are missing from the explicit lists, treat them as code to avoid loss.
        remaining_keys = [
            k
            for k in source_dict.keys()
            if k not in code_keys and k not in markdown_keys
        ]
        code_keys.extend([k for k in remaining_keys if k not in markdown_keys])
        markdown_keys.extend([k for k in remaining_keys if k not in code_keys])

        # Sort markdown cells by descending source length (longer markdown tends to appear earlier).
        markdown_keys.sort(key=lambda k: len(source_dict.get(k, "")), reverse=True)

        # Interleave code and markdown cells.
        interleaved = []
        i = 0
        while i < len(code_keys) or i < len(markdown_keys):
            if i < len(code_keys):
                interleaved.append(code_keys[i])
            if i < len(markdown_keys):
                interleaved.append(markdown_keys[i])
            i += 1
        final_order = interleaved

    predicted_orders[json_path.stem] = " ".join(final_order)



In [5]:
submission_df = pd.read_csv(SAMPLE_SUBMISSION)
submission_df["cell_order"] = submission_df["id"].map(predicted_orders).fillna("")
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
